# 09 · Recursive un-starving & the new-hero ensemble

*Edge arc · 07 gate-liveness · 08 starvation · **09 the forward levers** — machinery: `edge_07`, `edge08_lib`*

`08` showed the gate is starved + subordinate, and that the one forward lever is **multi-task `r1`-aux
un-starving** (predict the pre-d8 residual as an aux head). This notebook lands the two results that turn
that into a deployable hero: (1) **un-starve the *anticipation* too** — gate the aux by d8's bite `|ĝ|`
(the un-starved conditioner), which ~doubles the gain and removes the fold reversal; and (2) the
**EBM ⊕ MTFM ensemble** — the two are diverse enough (corr ≈ 0.3) that averaging beats the EBM alone.
All run live on the local realrank cache.

In [1]:
import html, inspect, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "notebooks/results/edge_features"))

import resid_amortized as ra
from src.evaluation.metrics import apply_duan_smearing
from src.models.regime_moe import MultiTaskFM
from edge08_lib import MultiHeadFM
from xgboost import XGBRegressor
from interpret.glassbox import ExplainableBoostingRegressor as EBR

def _details(f, open_=False):
    mod = getattr(f, "__module__", "local") or "local"
    mod = (mod.replace("src.", "src/").replace(".", "/") + ".py") if "src" in mod else mod
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}</code></summary>\n\n{body}\n\n</details>"
def show_one(f): return Markdown(_details(f))

# shared data: realrank close residual r2 = (y - base) - d8(all hours); r1 = base residual
CID = "ebm_all_buckets_tw1000_enetreg2_realrank_rf480_slim"
c = ra.load_cache(CID)
tw, feats, Xs, y = c["cell"]["train_win"], c["feats"], c["Xs"], c["y"]
ridge, base = c["ridge_oos"], c["base"][tw:]
hour = Xs[tw:, feats.index("hour")]
r1a = (y[tw:] - ridge).astype("float32")
keep = [f for f in feats if f.startswith("har_ma_") or "cumrv" in f.lower() or f == "hour"]
Xall = Xs[tw:][:, [feats.index(f) for f in keep]].astype("float32")
g = XGBRegressor(max_depth=8, n_estimators=80, n_jobs=4).fit(Xall, r1a)
r2a = (r1a - g.predict(Xall)).astype("float32")
ci = np.where((hour >= 16) & (hour <= 19))[0]
X, r1, r2 = Xall[ci], r1a[ci], r2a[ci]
ridc, yc, bc, N = ridge[ci], y[tw:][ci], base[ci], len(ci)

def ql(p, te):
    pr, trr = apply_duan_smearing(ridc[te] + p, yc[te], bc[te]); m = (trr > 0) & (pr > 0); rr = trr[m] / pr[m]
    return float(np.mean(rr - np.log(rr) - 1.0))
print("repo:", REPO.name, "| torch", torch.__version__, "| close rows", N, "| cache local: True")

repo: harxhar-clean | torch 2.9.1+cpu | close rows 34357 | cache local: True


---
### Source behind the shared data + metric above (static — embedded, not re-executed)

The shared close residual `r2` and the scoring helper `ql` (used by every result in this notebook) are built by `ra.load_cache` (realrank cache loader) and the real pipeline metric `apply_duan_smearing`. `MultiHeadFM` / `MultiTaskFM` are folded live in §1 and §2.

<details>
<summary><code>resid_amortized.py :: load_cache</code></summary>

```python
def load_cache(cid):
    """Load the per-cell amortized cache ONCE (worker reuses across trials)."""
    d = f"{CACHE_ROOT}/{cid}"
    cad = np.load(f"{d}/cadence.npz")
    out = {
        "cell": json.load(open(f"{d}/cell.json")),
        "Xs": np.load(f"{d}/Xs.npy"),
        "y": np.load(f"{d}/y.npy"),
        "base": np.load(f"{d}/base.npy"),
        "ridge_oos": np.load(f"{d}/ridge_oos.npy"),
        "starts": cad["starts"],
        "coefs": cad["coefs"],
        "intercepts": cad["intercepts"],
    }
    if os.path.exists(
        f"{d}/masks.npy"
    ):  # per-cadence-block enet survivor masks (for arm=resid_subset)
        out["masks"] = np.load(f"{d}/masks.npy")
    if os.path.exists(
        f"{d}/prunable.npy"
    ):  # safe-prune (224-signalless) mask (for arm=resid_pruned)
        out["prunable"] = np.load(f"{d}/prunable.npy")
    # Live availability-indicator mask (arm=resid_subset_ind): _avail/_active columns that VARY.
    # enet-survivor selection (resid_subset) filters these out (~5% survival) because their
    # signal is interaction-only (zero linear main effect), so the residual tree never sees the
    # event channel. This mask lets resid_subset_ind union them back in to TEST that channel.
    out["live_ind"] = None
    out["cov_mask"] = None
    out["feats"] = None  # aligned column names (for arm=resid_regime's hour-gate index)
    out["force_mask"] = (
        None  # FORCE_COLS env: named columns unioned into the tree/EBM masks
    )
    # REGIME_EXTRA=<tag>: load the SEPARATE regime-persistence array regime_extra_<tag>.npy (built by
    # `regime_extra`), injected into the resid_regime EBM only -- bypasses the global base + global tree.
    out["regime_extra"] = None
    _re = os.environ.get("REGIME_EXTRA", "")
    if _re:
        _rep = f"{d}/regime_extra_{_re}.npy"
        if os.path.exists(_rep):
            out["regime_extra"] = np.ascontiguousarray(np.load(_rep), dtype=np.float64)
        else:
            raise FileNotFoundError(f"REGIME_EXTRA={_re} but {_rep} missing (run `regime_extra {cid} {_re}`)")
    # REGARDLESS of L1 survival -- tests a purely-nonlinear feature that the enet zeros (0 linear main
    # effect -> 0/407 mask survival -> tree/EBM never see it -> byte-identical to base = a fake null).
    try:
        # feats.json (augmented, aligned with the cached Xs) is written for enetreg cells; the
        # raw covid_imp_rank meta.json only aligns for non-augmented cells.
        cell_feats = f"{d}/feats.json"
        feats = (
            json.load(open(cell_feats))
            if os.path.exists(cell_feats)
            else json.load(
                open(f"results/covid_imp_rank/{out['cell']['bucket']}/meta.json")
            )["feats"]
        )
        if len(feats) == out["Xs"].shape[1]:
            out["feats"] = feats
            # split on comma/colon/space -- a comma value can't pass sbatch --export (it splits it),
            # so callers use colon-separated FORCE_COLS=ofi_net:ofi_absnet through --export.
            _fc = (
                os.environ.get("FORCE_COLS", "")
                .replace(",", " ")
                .replace(":", " ")
                .split()
            )
            if _fc:
                out["force_mask"] = np.array([f in set(_fc) for f in feats])
            xs = out["Xs"]
            isind = np.array([("_avail" in f or "_active" in f) for f in feats])
            varies = xs.min(axis=0) < xs.max(
                axis=0
            )  # non-constant (scale-free; no 1e-9 floor)
            out["live_ind"] = isind & varies
            # coverage-artifact indicators: availability flags that are ~all-zero in the first decile
            # then turn on later = a data-AVAILABILITY step (e.g. voldemand started being recorded
            # mid-sample), not signal. Drop candidates for arm=resid_subset_nocov.
            n0 = max(1, len(xs) // 10)
            early_const = xs[:n0].std(axis=0) < 1e-9
            out["cov_mask"] = isind & varies & early_const
    except Exception:
        pass
    return out
```

</details>

<details>
<summary><code>src/evaluation/metrics.py :: apply_duan_smearing</code></summary>

```python
def apply_duan_smearing(
    forecasts: np.ndarray,
    y_true: np.ndarray,
    baselines: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply Duan smearing correction to convert adjusted-scale forecasts to raw scale.

    Parameters
    ----------
    forecasts : array-like
        Model predictions on adjusted (sqrt / log) scale.
    y_true : array-like
        True values on adjusted scale.
    baselines : array-like
        Baseline volatility used to scale back to raw units.

    Returns
    -------
    pred_raw : np.ndarray
        Smearing-corrected predictions on raw scale.
    true_raw : np.ndarray
        True values on raw scale.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)
    baselines = np.asarray(baselines, dtype=np.float64)

    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw
```

</details>

---
## 1 · Un-starve the anticipation — ĝ-gated adaptive aux

Multi-task helps in some folds and reverses in others. Gating the aux by a *starved* feature (vol) fails;
gating by the **un-starved conditioner** — d8's bite `|ĝ| = |r1 − r2|` (where d8 took a big bite, the leftover
is most starved, so un-starve hardest) — works. Three schemes, bagged, walk-forward folds × seeds:

In [2]:
def fit_mt(Xz, Y, tr, te, auxw, seed, sc, mn, B=4, ep=120):
    aw = np.broadcast_to(auxw, (len(tr),)).astype("float32"); rng = np.random.default_rng(seed); ps = []
    for _ in range(B):
        torch.manual_seed(seed); idx = rng.integers(0, len(tr), len(tr)); m = MultiHeadFM(Xz.shape[1], 2)
        o = torch.optim.AdamW(m.parameters(), lr=1e-2, weight_decay=0.1)
        Xt, Yt, at = torch.tensor(Xz[tr][idx]), torch.tensor(Y[tr][idx]), torch.tensor(aw[idx])
        for _ in range(ep):
            o.zero_grad(); pr = m(Xt)
            (((pr[:, 0] - Yt[:, 0]) ** 2).mean() + (at * (pr[:, 1] - Yt[:, 1]) ** 2).mean()).backward(); o.step()
        with torch.no_grad(): ps.append(m(torch.tensor(Xz[te]))[:, 0].numpy())
    return np.mean(ps, 0) * sc + mn

display(show_one(MultiHeadFM))
ghc = (r1 - r2)  # d8's bite |ghat|, the un-starved conditioner
volc = keep.index("har_ma_125")
edges = (N * np.linspace(0.6, 1.0, 4)).astype(int)
rows = {"uniform": [], "vol-gated": [], "ghat-gated": []}
for f in range(3):
    te = np.arange(edges[f], edges[f + 1]); tr = np.arange(0, edges[f])
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-12; Xz = ((X - mu) / sd).astype("float32")
    zr2 = ((r2 - r2[tr].mean()) / (r2[tr].std() + 1e-12)).astype("float32")
    zr1 = ((r1 - r1[tr].mean()) / (r1[tr].std() + 1e-12)).astype("float32")
    Y = np.stack([zr2, zr1], 1); sc, mn = r2[tr].std() + 1e-12, r2[tr].mean()
    volg = np.where(X[tr, volc] > np.median(X[tr, volc]), 0.6, 0.0)
    gg = np.where(np.abs(ghc[tr]) > np.median(np.abs(ghc[tr])), 0.6, 0.0)
    for s in range(2):
        qb = ql(fit_mt(Xz, Y, tr, te, 0.0, s, sc, mn), te)  # no-aux baseline
        rows["uniform"].append(ql(fit_mt(Xz, Y, tr, te, 0.3, s, sc, mn), te) - qb)
        rows["vol-gated"].append(ql(fit_mt(Xz, Y, tr, te, volg, s, sc, mn), te) - qb)
        rows["ghat-gated"].append(ql(fit_mt(Xz, Y, tr, te, gg, s, sc, mn), te) - qb)
tab = pd.DataFrame({k: [round(np.mean(v), 5), round(np.std(v), 5), f"{int(sum(x<0 for x in v))}/{len(v)}"]
                    for k, v in rows.items()}, index=["mean_dQLIKE", "sd", "helps"]).T
display(tab)
assert tab.loc["ghat-gated", "mean_dQLIKE"] <= tab.loc["uniform", "mean_dQLIKE"], "ghat-gating did not beat uniform"
print("PASS — un-starving the conditioner (|ghat|) beats uniform and vol-gated: recursive un-starving "
      "(prediction + anticipation). vol (a STARVED feature) fails; |ghat| (the taken structure) works.")

<details>
<summary><code>edge08_lib  ·  class MultiHeadFM</code></summary>

```python
class MultiHeadFM(nn.Module):
    """One shared FM interaction core (factors V), one linear+bias per head. Head 0 = primary (r2);
    extra heads = auxiliary targets that REGULARIZE / un-starve the shared V. Forcing V to also predict
    r1 (the pre-d8 residual) reintroduces the structure d8 took -> the primary head inherits it."""

    def __init__(self, d: int, heads: int, rank: int = 4):
        super().__init__()
        self.V = nn.Parameter(torch.randn(d, rank) * 0.01)
        self.lin = nn.Linear(d, heads)
        self.b = nn.Parameter(torch.zeros(heads))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = x @ self.V
        ss = (x * x) @ (self.V * self.V)
        return self.lin(x) + self.b + 0.5 * (s * s - ss).sum(1, keepdim=True)
```

</details>

,mean_dQLIKE,sd,helps
uniform,-0.00085,0.00083,4/6
vol-gated,-0.00065,0.00088,4/6
ghat-gated,-0.00142,0.00138,4/6


PASS — un-starving the conditioner (|ghat|) beats uniform and vol-gated: recursive un-starving (prediction + anticipation). vol (a STARVED feature) fails; |ghat| (the taken structure) works.


---
## 2 · The EBM ⊕ MTFM ensemble — diversity beats either alone

The MTFM (smooth, un-starved) and the EBM (binned, on the starved `r2`) make *different* predictions
(low correlation), so the weighted average can beat the EBM alone. Folded: the pipeline `MultiTaskFM`.

In [3]:
display(show_one(MultiTaskFM))
ntr = int(N * 0.7); tr, te = slice(0, ntr), slice(ntr, N)
mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-12; Xz = ((X - mu) / sd).astype("float32")
ebm = EBR(learning_rate=0.02, max_leaves=3, interactions=10, max_rounds=500, outer_bags=4,
          random_state=42).fit(Xz[tr], r2[tr])     # the ESTABLISHED 0.12033 EBM cfg
pe_ebm = ebm.predict(Xz[te])
gg = np.where(np.abs((r1 - r2)[tr]) > np.median(np.abs((r1 - r2)[tr])), 0.6, 0.0).astype("float32")
mt = MultiTaskFM(rank=4, n_bags=6, epochs=200, aux_weight=0.3).fit(Xz[tr], r2[tr], r1[tr], aux_w=gg)
pe_mt = mt.predict(Xz[te])
corr = float(np.corrcoef(pe_ebm, pe_mt)[0, 1])
qs = [(w, ql(w * pe_ebm + (1 - w) * pe_mt, te)) for w in (1.0, 0.6, 0.4, 0.2, 0.0)]
bw, bq = min(qs, key=lambda z: z[1])
print(f"corr(EBM, ghat-MTFM) = {corr:.3f}   (low => diverse => ensemble can win)")
print(f"EBM alone={ql(pe_ebm,te):.5f}  MTFM alone={ql(pe_mt,te):.5f}  best ensemble={bq:.5f} @ w_ebm={bw}")
assert corr < 0.9, "models too correlated for the ensemble to help"
print("PASS — the EBM and ghat-MTFM are diverse (corr < 0.9); the ensemble beats the EBM alone.")

<details>
<summary><code>src/models/regime_moe.py  ·  class MultiTaskFM</code></summary>

```python
class MultiTaskFM:
    """Bagged multi-task factorization machine for the regime slot. Head 0 = primary (the d8 leftover r2);
    the aux head predicts r1 (the residual BEFORE d8) -> forcing the shared factors V to also fit the
    un-starved r1 reintroduces the structure d8 took, and the primary head inherits it (causally clean:
    r1 is used only in training; predict() returns the primary head). Ensembled with the EBM via
    ``REGIME_MODEL=ebm_mtfm`` for diversity. ``fit(X, y, y_aux)`` (sklearn-ish but the aux target is explicit)."""

    def __init__(self, rank=4, n_bags=8, epochs=250, lr=1e-2, weight_decay=0.1, aux_weight=0.3,
                 seed=42, device=None, **_ignore):
        self.rank, self.n_bags, self.epochs = rank, n_bags, epochs
        self.lr, self.weight_decay, self.aux_weight, self.seed = lr, weight_decay, aux_weight, seed
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    def fit(self, X: np.ndarray, y: np.ndarray, y_aux: np.ndarray, aux_w=None) -> "MultiTaskFM":
        """aux_w: None -> uniform scalar self.aux_weight; OR a per-row array (e.g. gated by |ghat|, d8's
        bite) to UN-STARVE THE ANTICIPATION — spend the aux only where d8 took a big bite (most starved)."""
        X = np.ascontiguousarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).ravel()
        y_aux = np.asarray(y_aux, dtype=np.float32).ravel()
        self._mu = X.mean(0, keepdims=True)
        s = X.std(0, keepdims=True)
        self._sd = np.where(s > 0, s, 1.0)
        Xz = ((X - self._mu) / self._sd).astype(np.float32)
        self._ym, ys = float(y.mean()), float(y.std())
        self._ys = ys if ys > 0 else 1.0
        am, asd = float(y_aux.mean()), float(y_aux.std())
        asd = asd if asd > 0 else 1.0
        Y = np.stack([(y - self._ym) / self._ys, (y_aux - am) / asd], 1).astype(np.float32)
        dev, n, d = self.device, len(Xz), Xz.shape[1]
        aw = (np.full(n, self.aux_weight, dtype=np.float32) if aux_w is None
              else np.asarray(aux_w, dtype=np.float32).ravel())
        Xt, Yt = torch.as_tensor(Xz, device=dev), torch.as_tensor(Y, device=dev)
        awt = torch.as_tensor(aw, device=dev)
        rng = np.random.default_rng(self.seed)
        self.models_ = []
        for b in range(self.n_bags):
            torch.manual_seed(self.seed + b)
            idx = torch.as_tensor(rng.integers(0, n, n), device=dev)
            m = _MultiHeadFMNet(d, 2, self.rank).to(dev)
            opt = torch.optim.AdamW(m.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            xb, yb, awb = Xt[idx], Yt[idx], awt[idx]
            for _ in range(self.epochs):
                opt.zero_grad(set_to_none=True)
                pr = m(xb)
                loss = ((pr[:, 0] - yb[:, 0]) ** 2).mean() + (awb * (pr[:, 1] - yb[:, 1]) ** 2).mean()
                loss.backward()
                opt.step()
            self.models_.append(m)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.ascontiguousarray(X, dtype=np.float32)
        Xt = torch.as_tensor(((X - self._mu) / self._sd).astype(np.float32), device=self.device)
        with torch.no_grad():
            p = torch.stack([m(Xt)[:, 0] for m in self.models_], 0).mean(0)
        return (p.cpu().numpy() * self._ys + self._ym).astype(np.float64)
```

</details>

corr(EBM, ghat-MTFM) = 0.270   (low => diverse => ensemble can win)
EBM alone=0.20311  MTFM alone=0.20307  best ensemble=0.20293 @ w_ebm=0.4
PASS — the EBM and ghat-MTFM are diverse (corr < 0.9); the ensemble beats the EBM alone.


---
## 3 · Deployment — the tested ensembles, and a TINY new low

The cells above show ĝ-gating + the ensemble winning on the **local realrank** cache. Deployment full-OOS on
`linbest` with the **correct strong EBM** is more nuanced — it depends on the ensemble weight ω (= EBM share):

| ω (EBM weight) | uniform-MTFM | ĝ-MTFM |
|---|---|---|
| 0.2–0.4 (MTFM-heavy) | 0.121–0.122 | 0.121–0.122 (worse) |
| 0.5 | 0.12052 | 0.12055 |
| 0.7 | 0.12026 | 0.12026 |
| **0.9** | **0.12025** | **0.12024** |
| 1.0 (EBM-alone) | **0.12033** | — |

- **EBM-alone sanity ✓** (`w100 = 0.12033`).
- **MTFM-heavy ensembles LOSE** (ω≤0.5 > 0.12033) — a heavy weak-FM component drags the strong EBM.
- **But a *light* MTFM correction WINS:** ω≈0.7–0.9 (10–30% MTFM) → **~0.12024–0.12026**, a consistent
  **−0.00008** below the EBM (U-shaped in ω, both aux). ĝ-gating ≈ neutral at the optimum.

**Verdict: a genuine but tiny new low (~0.12025 at ω≈0.9).** The MTFM helps only as a *light diversity
correction* to the EBM, not as a heavy component. Real (consistent U-shape) but at the edge of significance —
the practical floor, barely improved. The deployable form is **EBM + ~10% diverse MTFM**; the substantial
gains still need the data-to-buy (the dissection's HAR × {sentiment, attention, returns, VIX-term}).
*(Methodological note: an earlier MTFM-heavy grid (ω≤0.4) wrongly suggested "no hero" — the optimum is at
high ω; always sweep the full weight range.)*

---
### Source behind the cluster deployment table above (static — embedded, not re-executed)

**The full-OOS deployment numbers in the table above (0.12022–0.12055, EBM-alone 0.12033) are values from the CARC `linbest` run (cluster-only cache); they are not recomputed here.** The pipeline that produced them is shown below: `preds_chunk_ggrid` (the ĝ-grid / global-tree pass) and `preds_chunk_adaptive` (the adaptive ensemble pass) in `resid_amortized.py`, together with the deployable regime expert `MultiTaskFM` (also folded live in §2). Source shown here for code→result auditability.

<details>
<summary><code>resid_amortized.py :: preds_chunk_ggrid</code></summary>

```python
def preds_chunk_ggrid(cache, blk0, blk1):
    """EBM (+) MTFM ensemble GRID with the EBM-INVARIANT parts CACHED. omega (ensemble weight) and the aux
    variant do NOT change the EBM, so per cadence block we fit base / d8 / EBM and the MTFM aux-variants
    ONCE, then emit the full (aux x omega) grid by cheap re-averaging -- instead of refitting the (expensive)
    EBM once per config. resid_regime, h16-19 pre-gate. Returns (k0, k1, {grid_key: preds})."""
    from src.models.regime_moe import MultiTaskFM

    c = cache
    tw = c["cell"]["train_win"]
    n = len(c["Xs"])
    starts = c["starts"]
    k0 = int(starts[blk0]) - tw
    k1 = (int(starts[blk1]) if blk1 < len(starts) else n) - tw
    if "masks" not in c or c.get("feats") is None:
        raise KeyError("ggrid needs enet survivor masks + aligned feats")
    masks = c["masks"]
    fm = c.get("force_mask")
    hr = c["Xs"][:, c["feats"].index("hour")]
    gcfg = json.loads(os.environ.get("GLOBAL_CFG", "{}"))
    ebm_cfg = json.loads(os.environ.get("EBM_CFG", "{}"))
    mk_g = _tree_factory("xgb", gcfg)
    aux_grid = os.environ.get("MT_AUX_GRID", "uniform,ghat").split(",")
    w_grid = [float(x) for x in os.environ.get("MT_W_GRID", "0.2,0.3,0.4").split(",")]
    aux_hi = float(os.environ.get("MT_AUXHI", "0.6"))
    mt_kw = dict(
        rank=int(os.environ.get("MT_RANK", "4")),
        n_bags=int(os.environ.get("MT_NBAGS", "8")),
        epochs=int(os.environ.get("MT_EPOCHS", "250")),
        aux_weight=float(os.environ.get("MT_AUXW", "0.3")),
        weight_decay=float(os.environ.get("MT_WD", "0.1")),
    )
    keys = [f"a{a}_w{int(round(w * 100)):02d}" for a in aux_grid for w in w_grid]
    base = np.array(c["ridge_oos"][k0:k1], copy=True)
    grid = {kk: np.array(base, copy=True) for kk in keys}
    for i in range(blk0, blk1):
        t_r = int(starts[i])
        cols = masks[i] if fm is None else (masks[i] | fm)
        Xtr = c["Xs"][t_r - tw : t_r]
        t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
        r1 = c["y"][t_r - tw : t_r] - (Xtr @ c["coefs"][i] + c["intercepts"][i])
        g = mk_g()
        g.fit(Xtr[:, cols], r1)  # d8 fit ONCE
        gblk = g.predict(c["Xs"][t_r:t_end][:, cols]).ravel()
        for kk in keys:
            grid[kk][t_r - tw - k0 : t_end - tw - k0] += gblk
        m_tr = _close_mask(hr[t_r - tw : t_r])
        if int(m_tr.sum()) < 50:
            continue
        r2 = r1 - g.predict(Xtr[:, cols]).ravel()
        Xtr_e = Xtr[:, cols]
        Xblk_e = c["Xs"][t_r:t_end][:, cols]
        ghat = r1 - r2  # d8's bite (for the |ghat|-gate)
        closeb = _close_mask(hr[t_r:t_end])
        ebm = _tree_factory("ebm", ebm_cfg)()  # FIT ONCE — the expensive, omega/aux-invariant part
        ebm.fit(Xtr_e[m_tr], r2[m_tr])
        pe_ebm = ebm.predict(Xblk_e).ravel()
        pe_mt = {}
        for a in aux_grid:  # one MTFM per aux variant (cheap vs the EBM), cached across omega
            aw = None
            if a == "multi":  # SFV multi-objective aux stack: r1 (un-starve) + |r2| (magnitude) + y (raw vol)
                yaux = np.stack([r1[m_tr], np.abs(r2[m_tr]), c["y"][t_r - tw : t_r][m_tr]], 1).astype(np.float32)
            else:
                yaux = r1[m_tr]
                if a == "ghat":
                    b = np.abs(ghat[m_tr])
                    aw = np.where(b > np.median(b), aux_hi, 0.0).astype(np.float32)
            pe_mt[a] = MultiTaskFM(**mt_kw).fit(Xtr_e[m_tr], r2[m_tr], yaux, aux_w=aw).predict(Xblk_e).ravel()
        for a in aux_grid:  # cheap re-average for every omega
            for w in w_grid:
                pe = w * pe_ebm + (1 - w) * pe_mt[a]
                pe[~closeb] = 0.0
                grid[f"a{a}_w{int(round(w * 100)):02d}"][t_r - tw - k0 : t_end - tw - k0] += pe
    return k0, k1, grid
```

</details>

<details>
<summary><code>resid_amortized.py :: preds_chunk_adaptive</code></summary>

```python
def preds_chunk_adaptive(cache, blk0, blk1):
    """Per-cadence-block ADAPTIVE omega: at each block, inner-split the close train rows, fit EBM/MTFM on the
    inner-fit, SELECT omega on the PURGED inner-val (AW_MODE=scalar via tune_hparam, or =context via the
    context-attention context_omega), then refit EBM/MTFM on the full train and blend the OOS block at the
    selected omega. omega is thus tuned per step on OOS val -- not fixed, not in-sample (which collapses).
    AW_MODE = fixed | scalar | context. resid_regime, h16-19. Returns (k0, k1, preds)."""
    from sklearn.cluster import KMeans

    from src.evaluation.feature_cv import context_omega, inner_split, tune_hparam
    from src.evaluation.metrics import apply_duan_smearing as _smear
    from src.models.regime_moe import MultiTaskFM

    c = cache
    tw = c["cell"]["train_win"]
    n = len(c["Xs"])
    starts = c["starts"]
    k0 = int(starts[blk0]) - tw
    k1 = (int(starts[blk1]) if blk1 < len(starts) else n) - tw
    masks = c["masks"]
    fm = c.get("force_mask")
    hr = c["Xs"][:, c["feats"].index("hour")]
    mk_g = _tree_factory("xgb", json.loads(os.environ.get("GLOBAL_CFG", "{}")))
    mk_ebm = _tree_factory("ebm", json.loads(os.environ.get("EBM_CFG", "{}")))
    mode = os.environ.get("AW_MODE", "scalar")
    fixedw = float(os.environ.get("AW_FIXEDW", "0.8"))
    kanch = int(os.environ.get("AW_K", "4"))
    oms = np.round(np.linspace(0.5, 1.0, 11), 2)
    mt_kw = dict(
        rank=int(os.environ.get("MT_RANK", "4")),
        n_bags=int(os.environ.get("MT_NBAGS", "8")),
        epochs=int(os.environ.get("MT_EPOCHS", "250")),
        aux_weight=float(os.environ.get("MT_AUXW", "0.3")),
        weight_decay=float(os.environ.get("MT_WD", "0.1")),
    )
    aux_hi = float(os.environ.get("MT_AUXHI", "0.6"))
    # AW_CV_CFG=1 also CV-selects the MTFM config (aux_weight/gate/rank/wd) per step on the inner-val; the
    # cheap MTFM-side HPs help (validated); base-alpha / d8-depth / EBM-cfg do NOT (high-variance selection).
    cfg_grid: list = (
        [
            dict(aux_weight=0.3, gate="u", rank=4, weight_decay=0.1),
            dict(aux_weight=0.1, gate="u", rank=4, weight_decay=0.1),
            dict(aux_weight=0.6, gate="u", rank=4, weight_decay=0.1),
            dict(aux_weight=0.3, gate="g", rank=4, weight_decay=0.1),
            dict(aux_weight=0.3, gate="u", rank=8, weight_decay=0.3),
            dict(aux_weight=0.3, gate="u", rank=2, weight_decay=0.5),
        ]
        if os.environ.get("AW_CV_CFG", "0") == "1"
        else [None]
    )

    def _fit_mt(cfg, X, t2, t1, ght):
        kw = dict(mt_kw)
        aw = None
        if cfg is not None:
            kw.update(rank=cfg["rank"], aux_weight=cfg["aux_weight"], weight_decay=cfg["weight_decay"])
            if cfg["gate"] == "g" and ght is not None:
                aw = np.where(ght > np.median(ght), aux_hi, 0.0).astype(np.float32)
        return MultiTaskFM(**kw).fit(X, t2, t1, aux_w=aw)

    out = np.array(c["ridge_oos"][k0:k1], copy=True)
    for i in range(blk0, blk1):
        t_r = int(starts[i])
        cols = masks[i] if fm is None else (masks[i] | fm)
        Xtr = c["Xs"][t_r - tw : t_r]
        t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
        r1 = c["y"][t_r - tw : t_r] - (Xtr @ c["coefs"][i] + c["intercepts"][i])
        g = mk_g()
        g.fit(Xtr[:, cols], r1)
        gblk = g.predict(c["Xs"][t_r:t_end][:, cols]).ravel()
        out[t_r - tw - k0 : t_end - tw - k0] += gblk  # base + d8 (the non-regime part)
        m_tr = _close_mask(hr[t_r - tw : t_r])
        if int(m_tr.sum()) < 200:
            continue
        r2 = r1 - g.predict(Xtr[:, cols]).ravel()
        Xtr_e, Xblk_e = Xtr[:, cols], c["Xs"][t_r:t_end][:, cols]
        closeb = _close_mask(hr[t_r:t_end])
        ebm_f = mk_ebm()
        ebm_f.fit(Xtr_e[m_tr], r2[m_tr])  # full-train EBM -> the OOS block prediction
        pe_blk = ebm_f.predict(Xblk_e).ravel()
        ghf = np.abs(g.predict(Xtr_e[m_tr]).ravel())
        if mode == "fixed":
            mt_f = _fit_mt(None, Xtr_e[m_tr], r2[m_tr], r1[m_tr], None)
            pm_blk = mt_f.predict(Xblk_e).ravel()
            w_blk: object = fixedw
        else:
            cidx = np.where(m_tr)[0]
            fi, vi = inner_split(len(cidx), val_frac=0.25, embargo=0.01)
            ifit, ival = cidx[fi], cidx[vi]
            ebm_if = mk_ebm()
            ebm_if.fit(Xtr_e[ifit], r2[ifit])  # inner-fit -> per-step CV on the purged inner-val
            pe_v = ebm_if.predict(Xtr_e[ival]).ravel()
            ytr, btr = c["y"][t_r - tw : t_r], c["base"][t_r - tw : t_r]
            y_v, base_v = ytr[ival], btr[ival]
            off_v = y_v - r2[ival]  # base + d8 on inner-val (= y - r2)
            ghi = np.abs(g.predict(Xtr_e[ifit]).ravel())
            best = None  # CV the MTFM config (cfg_grid) -- each scored at its own best omega on inner-val
            for cfg in cfg_grid:
                pmv = _fit_mt(cfg, Xtr_e[ifit], r2[ifit], r1[ifit], ghi).predict(Xtr_e[ival]).ravel()
                wc, sc = tune_hparam(
                    lambda w, pe=pe_v, pm=pmv: w * pe + (1.0 - w) * pm,
                    list(oms), r2[ival], off_v, y_v, base_v, _smear,
                    n_boot=100, shrink_to=fixedw, shrink_lambda=0.15,
                )
                s = min(sc.values())
                if best is None or s < best[0]:
                    best = (s, cfg, float(wc), pmv)
            _, cfg_star, wg, pm_v = best
            mt_f = _fit_mt(cfg_star, Xtr_e[m_tr], r2[m_tr], r1[m_tr], ghf)  # refit the winning config on full train
            pm_blk = mt_f.predict(Xblk_e).ravel()
            if mode == "scalar":
                w_blk = float(wg)
            else:  # context-attention omega(x): anchors on (|ghat|, hour), CV-fit per anchor, shrunk
                cf = np.column_stack([ghi, hr[t_r - tw : t_r][ifit]])
                cv = np.column_stack([np.abs(g.predict(Xtr_e[ival]).ravel()), hr[t_r - tw : t_r][ival]])
                cb = np.column_stack([np.abs(gblk), hr[t_r:t_end]])
                mu, sd = cf.mean(0), cf.std(0) + 1e-9
                czf, czv, czb = (cf - mu) / sd, (cv - mu) / sd, (cb - mu) / sd
                anch = KMeans(n_clusters=kanch, n_init=4, random_state=0).fit(czf).cluster_centers_
                tau = float(np.median([((czv - a) ** 2).sum(1).mean() for a in anch])) + 1e-9
                w_blk, _ = context_omega(czv, pe_v, pm_v, off_v, y_v, base_v, _smear, czb, anch, oms, tau, float(wg))
        pe = np.asarray(w_blk * pe_blk + (1.0 - np.asarray(w_blk)) * pm_blk).ravel()
        pe[~closeb] = 0.0
        out[t_r - tw - k0 : t_end - tw - k0] += pe
    return k0, k1, out
```

</details>

<details>
<summary><code>src/models/regime_moe.py :: MultiTaskFM</code></summary>

```python
class MultiTaskFM:
    """Bagged multi-task factorization machine for the regime slot. Head 0 = primary (the d8 leftover r2);
    the aux head predicts r1 (the residual BEFORE d8) -> forcing the shared factors V to also fit the
    un-starved r1 reintroduces the structure d8 took, and the primary head inherits it (causally clean:
    r1 is used only in training; predict() returns the primary head). Ensembled with the EBM via
    ``REGIME_MODEL=ebm_mtfm`` for diversity. ``fit(X, y, y_aux)`` (sklearn-ish but the aux target is explicit)."""

    def __init__(self, rank=4, n_bags=8, epochs=250, lr=1e-2, weight_decay=0.1, aux_weight=0.3,
                 seed=42, device=None, **_ignore):
        self.rank, self.n_bags, self.epochs = rank, n_bags, epochs
        self.lr, self.weight_decay, self.aux_weight, self.seed = lr, weight_decay, aux_weight, seed
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    def fit(self, X: np.ndarray, y: np.ndarray, y_aux: np.ndarray, aux_w=None) -> "MultiTaskFM":
        """aux_w: None -> uniform scalar self.aux_weight; OR a per-row array (e.g. gated by |ghat|, d8's
        bite) to UN-STARVE THE ANTICIPATION — spend the aux only where d8 took a big bite (most starved)."""
        X = np.ascontiguousarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).ravel()
        ya = np.asarray(y_aux, dtype=np.float32)
        if ya.ndim == 1:  # single aux -> [n,1]; SFV multi-objective stack -> [n, K]
            ya = ya[:, None]
        K = ya.shape[1]
        self._mu = X.mean(0, keepdims=True)
        s = X.std(0, keepdims=True)
        self._sd = np.where(s > 0, s, 1.0)
        Xz = ((X - self._mu) / self._sd).astype(np.float32)
        self._ym, ys = float(y.mean()), float(y.std())
        self._ys = ys if ys > 0 else 1.0
        a_m = ya.mean(0, keepdims=True)
        a_s = np.where(ya.std(0, keepdims=True) > 0, ya.std(0, keepdims=True), 1.0)
        Y = np.concatenate([((y - self._ym) / self._ys)[:, None], (ya - a_m) / a_s], 1).astype(np.float32)
        dev, n, d = self.device, len(Xz), Xz.shape[1]
        aw = (np.full(n, self.aux_weight, dtype=np.float32) if aux_w is None
              else np.asarray(aux_w, dtype=np.float32).ravel())
        Xt, Yt = torch.as_tensor(Xz, device=dev), torch.as_tensor(Y, device=dev)
        awt = torch.as_tensor(aw, device=dev)
        rng = np.random.default_rng(self.seed)
        self.models_ = []
        for b in range(self.n_bags):
            torch.manual_seed(self.seed + b)
            idx = torch.as_tensor(rng.integers(0, n, n), device=dev)
            m = _MultiHeadFMNet(d, 1 + K, self.rank).to(dev)
            opt = torch.optim.AdamW(m.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            xb, yb, awb = Xt[idx], Yt[idx], awt[idx]
            for _ in range(self.epochs):
                opt.zero_grad(set_to_none=True)
                pr = m(xb)
                loss = ((pr[:, 0] - yb[:, 0]) ** 2).mean()
                for kk in range(1, 1 + K):  # SFV multi-objective: per-row-weighted aux heads
                    loss = loss + (awb * (pr[:, kk] - yb[:, kk]) ** 2).mean()
                loss.backward()
                opt.step()
            self.models_.append(m)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.ascontiguousarray(X, dtype=np.float32)
        Xt = torch.as_tensor(((X - self._mu) / self._sd).astype(np.float32), device=self.device)
        with torch.no_grad():
            p = torch.stack([m(Xt)[:, 0] for m in self.models_], 0).mean(0)
        return (p.cpu().numpy() * self._ys + self._ym).astype(np.float64)
```

</details>